# 03 - Entregas, atrasos e satisfacao

Aqui eu junto tempo de entrega e reviews para entender onde atraso pesa mais na experiencia do cliente.

In [ ]:
from pathlib import Path
import sqlite3
import urllib.request
import zipfile

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 120)

PROJECT_DIR = Path.cwd()
DB_ZIP_URL = "https://github.com/Urpia-S/Olist_E-commerce_Analytic-SQL-Python/releases/download/data-v1/olist_colab.sqlite.zip"
OUTPUT_DIR = PROJECT_DIR / "outputs_colab"
DB_PATH = PROJECT_DIR / "olist_colab.sqlite"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def baixar_banco_da_release():
    zip_path = PROJECT_DIR / "olist_colab.sqlite.zip"
    urllib.request.urlretrieve(DB_ZIP_URL, zip_path)

    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(PROJECT_DIR)

    print("Banco extraido em:", DB_PATH)


if not DB_PATH.exists():
    baixar_banco_da_release()

conn = sqlite3.connect(DB_PATH)


def consulta(sql):
    return pd.read_sql_query(sql, conn)


def salvar_consulta(sql, arquivo):
    df = consulta(sql)
    destino = OUTPUT_DIR / arquivo
    df.to_csv(destino, index=False)
    print(f"Arquivo salvo: {destino}")
    return df


def grafico_barras(df, x, y, titulo, rotacao=0, top=None):
    dados = df.head(top) if top else df
    ax = dados.plot(kind="bar", x=x, y=y, legend=False, figsize=(10, 4))
    ax.set_title(titulo)
    ax.set_xlabel("")
    ax.set_ylabel(y)
    plt.xticks(rotation=rotacao, ha="right" if rotacao else "center")
    plt.tight_layout()
    plt.show()


consulta("""
-- Objetos disponiveis no banco preparado.
SELECT
    type AS tipo,
    name AS objeto
FROM sqlite_master
WHERE type IN ('table', 'view')
ORDER BY type, name
LIMIT 20;
""")

## 1. Entregas e atrasos

Calculo indicadores por estado e por categoria usando a view de entrega.

In [ ]:
indicadores_entrega_estado = salvar_consulta("""
-- Indicadores de entrega por estado do cliente.
SELECT
    customer_state AS estado_cliente,
    COUNT(*) AS pedidos,
    SUM(CASE WHEN order_delivered_customer_date IS NOT NULL THEN 1 ELSE 0 END) AS pedidos_entregues,
    ROUND(AVG(dias_entrega), 2) AS media_dias_entrega,
    ROUND(AVG(dias_ate_aprovacao), 2) AS media_dias_ate_aprovacao,
    ROUND(AVG(dias_ate_transportadora), 2) AS media_dias_ate_transportadora,
    ROUND(AVG(dias_transporte_cliente), 2) AS media_dias_transporte_cliente,
    ROUND(AVG(dias_atraso), 2) AS media_dias_atraso,
    ROUND(
        100.0 * SUM(CASE WHEN entrega_atrasada = 1 THEN 1 ELSE 0 END)
        / NULLIF(SUM(CASE WHEN order_delivered_customer_date IS NOT NULL THEN 1 ELSE 0 END), 0),
        2
    ) AS percentual_atraso
FROM vw_order_delivery
GROUP BY customer_state
ORDER BY percentual_atraso DESC, pedidos DESC;
""", "indicadores_entrega_estado.csv")

indicadores_entrega_estado.head(10)

In [ ]:
grafico_barras(indicadores_entrega_estado, "estado_cliente", "percentual_atraso", "Estados com maior percentual de atraso", rotacao=0, top=10)

In [ ]:
atraso_por_categoria = salvar_consulta("""
-- Atraso por categoria, filtrando categorias com volume minimo.
SELECT
    i.categoria_ingles,
    COUNT(DISTINCT d.order_id) AS pedidos,
    ROUND(AVG(d.dias_entrega), 2) AS media_dias_entrega,
    ROUND(AVG(d.dias_atraso), 2) AS media_dias_atraso,
    ROUND(
        100.0 * COUNT(DISTINCT CASE WHEN d.entrega_atrasada = 1 THEN d.order_id END)
        / NULLIF(COUNT(DISTINCT CASE WHEN d.order_delivered_customer_date IS NOT NULL THEN d.order_id END), 0),
        2
    ) AS percentual_atraso
FROM vw_order_delivery d
JOIN vw_order_items_enriched i ON i.order_id = d.order_id
GROUP BY i.categoria_ingles
HAVING COUNT(DISTINCT d.order_id) >= 30
ORDER BY percentual_atraso DESC, pedidos DESC;
""", "atraso_por_categoria.csv")

atraso_por_categoria.head()

In [ ]:

atraso_por_vendedor = salvar_consulta("""
-- Atraso por vendedor, considerando pedidos entregues e vendedores com volume minimo.
WITH vendedor_pedido AS (
    SELECT DISTINCT
        seller_id,
        order_id
    FROM vw_order_items_enriched
)
SELECT
    vp.seller_id,
    COUNT(DISTINCT d.order_id) AS pedidos,
    ROUND(AVG(d.dias_atraso), 2) AS media_dias_atraso,
    ROUND(
        100.0 * COUNT(DISTINCT CASE WHEN d.entrega_atrasada = 1 THEN d.order_id END)
        / NULLIF(COUNT(DISTINCT CASE WHEN d.order_delivered_customer_date IS NOT NULL THEN d.order_id END), 0),
        2
    ) AS percentual_atraso
FROM vendedor_pedido vp
JOIN vw_order_delivery d ON d.order_id = vp.order_id
GROUP BY vp.seller_id
HAVING COUNT(DISTINCT d.order_id) >= 20
ORDER BY percentual_atraso DESC, pedidos DESC;
""", "atraso_por_vendedor.csv")

atraso_por_vendedor.head(10)


## 2. Satisfacao

Depois comparo notas de review com status de entrega, estado e categoria.

In [ ]:
distribuicao_reviews = salvar_consulta("""
-- Distribuicao das notas dos reviews.
SELECT
    review_score,
    CASE
        WHEN review_score IN (1, 2) THEN 'baixa'
        WHEN review_score = 3 THEN 'neutra'
        WHEN review_score IN (4, 5) THEN 'alta'
    END AS classe_satisfacao,
    COUNT(*) AS reviews
FROM core_order_reviews
GROUP BY review_score
ORDER BY review_score;
""", "distribuicao_reviews.csv")

satisfacao_por_status_entrega = salvar_consulta("""
-- Comparo nota media e reviews baixos por status de entrega.
SELECT
    CASE
        WHEN d.entrega_atrasada = 1 THEN 'atrasada'
        WHEN d.entrega_atrasada = 0 THEN 'no_prazo'
        ELSE 'sem_entrega_confirmada'
    END AS status_entrega,
    COUNT(DISTINCT d.order_id) AS pedidos,
    ROUND(AVG(r.nota_media_review), 2) AS nota_media,
    ROUND(100.0 * SUM(r.reviews_baixos) / NULLIF(SUM(r.quantidade_reviews), 0), 2) AS percentual_reviews_baixos
FROM vw_order_delivery d
LEFT JOIN vw_reviews_por_pedido r ON r.order_id = d.order_id
GROUP BY status_entrega
ORDER BY nota_media;
""", "satisfacao_por_status_entrega.csv")

display(distribuicao_reviews)
display(satisfacao_por_status_entrega)

In [ ]:

satisfacao_por_estado = salvar_consulta("""
-- Satisfacao por estado do cliente.
SELECT
    d.customer_state AS estado_cliente,
    COUNT(DISTINCT d.order_id) AS pedidos_avaliados,
    ROUND(AVG(r.nota_media_review), 2) AS nota_media,
    ROUND(100.0 * SUM(r.reviews_baixos) / NULLIF(SUM(r.quantidade_reviews), 0), 2) AS percentual_reviews_baixos
FROM vw_order_delivery d
JOIN vw_reviews_por_pedido r ON r.order_id = d.order_id
GROUP BY d.customer_state
ORDER BY nota_media ASC, percentual_reviews_baixos DESC;
""", "satisfacao_por_estado.csv")

satisfacao_por_estado.head(10)


In [ ]:
grafico_barras(satisfacao_por_status_entrega, "status_entrega", "nota_media", "Nota media por status de entrega", rotacao=20)

In [ ]:
satisfacao_por_categoria = salvar_consulta("""
-- Satisfacao por categoria com volume minimo de pedidos avaliados.
SELECT
    i.categoria_ingles,
    COUNT(DISTINCT i.order_id) AS pedidos_avaliados,
    ROUND(AVG(r.nota_media_review), 2) AS nota_media,
    ROUND(100.0 * SUM(r.reviews_baixos) / NULLIF(SUM(r.quantidade_reviews), 0), 2) AS percentual_reviews_baixos
FROM vw_order_items_enriched i
JOIN vw_reviews_por_pedido r ON r.order_id = i.order_id
GROUP BY i.categoria_ingles
HAVING COUNT(DISTINCT i.order_id) >= 30
ORDER BY percentual_reviews_baixos DESC, pedidos_avaliados DESC;
""", "satisfacao_por_categoria.csv")

satisfacao_por_categoria.head(10)